# 備戰工作簿 · 卡片動作的價格分布與掛單落點

## 這個 notebook 不會下單

它**不連券商、不取即時報價、不產生任何委託**。這不是省略，是這條支線的契約：
`AI_HANDOFF.md` 寫著「不得讀取 credentials、登入券商或實際傳送／修改／取消委託」，
每一支腳本的檔頭都有 `No network, no broker, no order path`。
一個抓價格的程式不應該有能力送出委託 —— 那是這整條支線之所以安全的原因。

它也**不建議買價**。哪一檔、幾塊錢、買多少，是你的決定。

## 它做什麼

1. 把卡片開出的每一個進出，配上這檔股票**實際成交過的價格分布**
2. 顯示你自己已成交的單**歷史上落在當日區間的哪裡**
3. 產出一張**空白的下單工作表** —— 價格欄留給你自己填
4. 事後把你實際的成交回填，量測這一輪的落點與履約落差

第 3 步是刻意留白的。這個工作簿把分布擺出來，價格由你決定。

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

import build_mainline2 as m2
import build_prep as prep

bars = m2.load_bars()
fills = m2.load_fills()
signal_asof, _ = m2.load_signals()
signals = prep.read(ROOT / "inputs" / "latest_strategy_signals.csv")
snapshot_day, held = prep.latest_holdings()
actions = prep.action_list(signals)

print(f"訊號日 {signal_asof} · 庫存快照 {snapshot_day}")
print(f"行情最新 {max(s[-1]['date'] for s in bars.values())}")
print(f"卡片動作 {len(actions)} 個（括號部位已排除）")

## 1 · 你的單歷史上落在哪裡

把每一筆已結算成交價，放回**當天自己的最高／最低區間**。
0% 是當日最低、100% 是當日最高。這是唯一從實際成交算出來的掛單位置證據。

In [ ]:
landing = prep.fill_landing_stats(fills, bars)

print(f"已成交買進 {landing['buys']} 筆，平均落點 {landing['buy_mean']:.0%}")
print(f"歷來最低落點 {landing['lowest']:.0%}")
if landing["sell_mean"] is not None:
    print(f"賣出 {landing['sells']} 筆，平均落點 {landing['sell_mean']:.0%}")
print()
print("買進落點分布（每 10% 一格）")
peak = max(landing["deciles"]) or 1
for index, count in enumerate(landing["deciles"]):
    bar = "█" * round(count / peak * 34)
    print(f"  {index * 10:>3d}-{index * 10 + 10:<3d}% {bar} {count or ''}")
print()
print("掛得越低買到越便宜，但成交機率越低。這個取捨的兩邊都在上面，怎麼選是你的決定。")

## 2 · 每一檔的價格分布

六個月分價（volume-at-price）：這檔股票的成交量實際堆在哪些價位。

- **POC** — 成交量最大的價位
- **價值區** — 涵蓋 70% 量能的價格帶
- **現價分位** — 有多少比例的成交量發生在最新收盤以下
- **日內區間中位數** — 近 10 個交易日，這檔自己一天走多寬

> 分價分布是**過去成交發生在哪裡**的紀錄，不是未來應該在哪裡買的預測。
> POC 是描述統計名詞，不是「合理價」。

In [ ]:
rows = []
for item in actions:
    series = bars.get(item["code"])
    if not series:
        print(f"  {item['code']} {item['name']}：無行情歷史，略過")
        continue
    profile = m2.volume_profile(series)
    last = series[-1]
    ranges = prep.recent_ranges(series)
    cap = m2.capacity(series, last["close"])
    rows.append({
        "動作": item["action"],
        "策略": m2.STRATEGY_LABELS.get(item["strategy_id"], item["strategy_id"]),
        "代碼": item["code"],
        "名稱": item["name"],
        "持有": "是" if item["code"] in held else "否",
        "最新收盤": last["close"],
        "現價分位": m2.percentile_of(profile, last["close"]),
        "POC": profile["poc"],
        "價值區低": profile["value_low"],
        "價值區高": profile["value_high"],
        "六月最低": profile["low"],
        "六月最高": profile["high"],
        "日內區間中位數%": ranges["median_span_pct"],
        "20日均量": cap["avg_volume"],
        "單量佔比": cap["participation"],
    })

for row in sorted(rows, key=lambda r: (r["動作"], r["代碼"])):
    print(f"{row['動作']} {row['策略']:4s} {row['代碼']} {row['名稱']:10s} 持有={row['持有']}")
    print(f"    收盤 {row['最新收盤']:>10,.2f}  分位 {row['現價分位']:>5.0%}  "
          f"POC {row['POC']:>10,.2f}")
    print(f"    價值區 {row['價值區低']:,.2f} – {row['價值區高']:,.2f}   "
          f"六月 {row['六月最低']:,.2f} – {row['六月最高']:,.2f}")
    print(f"    日內區間中位數 {row['日內區間中位數%']:.1f}%   "
          f"20日均量 {row['20日均量']:,.0f}   單量佔 {row['單量佔比']:.3%}")
    print()

## 3 · 下單工作表（價格欄留白）

產出一張 CSV，每一個卡片動作一列。**`my_limit_price` 是空的，由你自己填。**

這裡不填價格不是功能沒做完 —— 是這個工作簿不做那件事。
上面的分布給你看，價格你決定，單你自己在券商介面下。

In [ ]:
import csv
from datetime import datetime

effective = next((a["effective"] for a in actions if a["effective"]), "")
target = ROOT / "output" / f"order_worksheet_{effective or 'next'}.csv"

fields = [
    "effective_date", "strategy_id", "stock_code", "stock_name", "action",
    "held", "last_close", "close_percentile", "poc", "value_low", "value_high",
    "median_intraday_range_pct", "avg_volume_20d",
    "my_limit_price", "my_shares", "my_note",
]
with target.open("w", encoding="utf-8-sig", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    for row in sorted(rows, key=lambda r: (r["動作"], r["代碼"])):
        writer.writerow({
            "effective_date": effective,
            "strategy_id": row["策略"],
            "stock_code": row["代碼"],
            "stock_name": row["名稱"],
            "action": row["動作"],
            "held": row["持有"],
            "last_close": f"{row['最新收盤']:g}",
            "close_percentile": f"{row['現價分位']:.4f}",
            "poc": f"{row['POC']:.2f}",
            "value_low": f"{row['價值區低']:.2f}",
            "value_high": f"{row['價值區高']:.2f}",
            "median_intraday_range_pct": f"{row['日內區間中位數%']:.2f}",
            "avg_volume_20d": f"{row['20日均量']:.0f}",
            "my_limit_price": "",
            "my_shares": "",
            "my_note": "",
        })

print(f"已產出 {target}")
print(f"{len(rows)} 列，my_limit_price / my_shares 留白等你填")
print()
print("填完之後，單還是你自己在券商介面下。這個檔案不會、也不能送出任何委託。")

## 4 · 事後回測你這一輪的掛單

成交回報進 `actual_fills.csv`、訊號配對進 `signal_fills.csv` 之後，
重跑這一格就能看到這一輪的落點與履約落差 —— 這才是「買得更低、賣得更高」
唯一能被驗證的版本：**事後量測，不是事前預測**。

In [ ]:
landings = m2.fill_landings(m2.load_fills(), m2.load_bars())
recent = [r for r in landings if r.get("position") is not None][-12:]

print("最近 12 筆成交的區間落點")
for row in recent:
    side = "買" if row["side"] == "BUY" else "賣"
    mark = "█" * round(row["position"] * 24)
    print(f"  {row['date']} {side} {row['stock_code']} {row['stock_name']:9s} "
          f"@{row['price']:>9,.2f}  {row['position']:>4.0%} |{mark:<24s}| "
          f"當日 {row['bar']['low']:g}–{row['bar']['high']:g}")

print()
print("樣本累積到 30 筆以上，平均落點才有統計意義，也才談得上要不要改掛單方式。")